# Bài 2 · kNN và chuẩn hoá đặc trưng · Thực hành trên lớp (v3)

Mở notebook từ thư mục bài. Corpus tiếng Anh lịch sử (UCI SMS Spam Collection). Ví dụ năm điểm là mô phỏng. Không chọn cấu hình bằng test.
Các tin nhắn chỉ dùng để học: không gọi số hoặc truy cập liên kết trong nội dung.

Quy ước kNN của bài: xếp theo d²; **đồng d² thì mã bản ghi nhỏ hơn đứng trước**. Mọi con số khớp `results/case_v3.json`.

In [1]:
from pathlib import Path
import sys, numpy as np, pandas as pd
ROOT = Path.cwd()
assert (ROOT / 'source/sms_case_v3.py').exists(), 'Mở notebook từ thư mục Bai02_v3'
sys.path.insert(0, str(ROOT / 'source'))
import sms_case_v3 as C
raw, d, tr, va, te = C.load_data()
print('Tệp gốc:', len(raw), '· sau bỏ trùng:', len(d), '· train/validation/test:', len(tr), len(va), len(te))

Tệp gốc: 5574 · sau bỏ trùng: 5159 · train/validation/test: 3095 1032 1032


## Chặng 1 · Từ tin nhắn đến một điểm

In [2]:
for ma in [2, 3, 4500]:
    r = d.loc[d.source_record == ma].iloc[0]
    print(ma, r.split, repr(r.text))
    print('   L =', int(r.n_chars), '· D =', int(r.n_digits), '· x =', (int(r.n_chars), round(float(r.digit_fraction), 4)))
assert d.loc[d.source_record == 3, 'n_digits'].iloc[0] == 25

2 train 'Ok lar... Joking wif u oni...'
   L = 29 · D = 0 · x = (29, 0.0)
3 train "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's"
   L = 155 · D = 25 · x = (155, 0.1613)
4500 validation 'Latest Nokia Mobile or iPOD MP3 Player +£400 proze GUARANTEED! Reply with: WIN to 83355 now! Norcorp Ltd.£1,50/Mtmsgrcvd18+'
   L = 123 · D = 14 · x = (123, 0.1138)


## Chặng 2 · kNN trên năm điểm mô phỏng

In [3]:
P = np.array([[0, 0], [2, 0], [0, 3], [4, 4], [1, 4]], float); nhan = np.array([0, 1, 1, 0, 0]); ma = np.arange(1, 6)
X = np.array([1., 1.])
d2 = C.sqdist(P, X)
print('d² =', d2.tolist())
for k in [1, 3, 5]:
    p, f, idx = C.knn(P, nhan, ma, X[None, :], k, return_index=True)
    print(f'k = {k}: láng giềng', [f'P{i + 1}' for i in idx[0]], '→', ['ham', 'spam'][p[0]])
assert list(d2) == [2, 2, 5, 18, 9]

d² = [2.0, 2.0, 5.0, 18.0, 9.0]
k = 1: láng giềng ['P1'] → ham
k = 3: láng giềng ['P1', 'P2', 'P3'] → spam
k = 5: láng giềng ['P1', 'P2', 'P3', 'P5', 'P4'] → ham


## Chặng 3 · μ, σ chỉ từ train; tin 4500 trước và sau chuẩn hoá

In [4]:
F = C.FEATURES
Xtr, ytr, rec = d.loc[tr, F].to_numpy(), d.loc[tr, 'y'].to_numpy(), d.loc[tr, 'source_record'].to_numpy()
mu, sd = Xtr.mean(0), Xtr.std(0)
print('μ =', mu.round(5).tolist(), '· σ =', sd.round(5).tolist())
q = d.loc[d.source_record == 4500, F].to_numpy()
for ten, sc in [('thô', None), ('chuẩn hoá', sd)]:
    p, f, idx = C.knn(Xtr, ytr, rec, q, 3, return_index=True, scale=sc)
    print(ten, '→ láng giềng', rec[idx[0]].tolist(), '→', ['ham', 'spam'][p[0]])

μ = [79.31018, 0.01805] · σ = [59.10396, 0.04706]
thô → láng giềng [2780, 626, 646] → ham
chuẩn hoá → láng giềng [4798, 3969, 2353] → spam


## Chặng 4 · Validation chọn cấu hình; test đã khoá

In [5]:
Xva, yva = d.loc[va, F].to_numpy(), d.loc[va, 'y'].to_numpy()
for sc in [False, True]:
    for k in [3, 5, 11]:
        p, _ = C.knn(Xtr, ytr, rec, Xva, k, scale=sd if sc else None)
        print('chuẩn hoá' if sc else 'thô     ', 'k =', k, '· tin sai:', int((p != yva).sum()))

thô      k = 3 · tin sai: 52


thô      k = 5 · tin sai: 54


thô      k = 11 · tin sai: 80


chuẩn hoá k = 3 · tin sai: 29


chuẩn hoá k = 5 · tin sai: 29


chuẩn hoá k = 11 · tin sai: 25


In [6]:
# Test chỉ chấm cấu hình đã khoá (chuẩn hoá, k = 11). Tệp dự đoán chuyển sang Bài 3.
pred = pd.read_csv(ROOT / 'data/sms_test_predictions_v3.csv')
yt, pt = pred.y.to_numpy(), pred.prediction.to_numpy()
cm = [[int(((pt == 0) & (yt == 0)).sum()), int(((pt == 1) & (yt == 0)).sum())], [int(((pt == 0) & (yt == 1)).sum()), int(((pt == 1) & (yt == 1)).sum())]]
print('Hàng: thật ham/spam · cột: dự đoán ham/spam:', cm)
assert cm == [[891, 13], [17, 111]]

Hàng: thật ham/spam · cột: dự đoán ham/spam: [[891, 13], [17, 111]]


## Phiếu cuối buổi · truy vấn X′ = (2; 1)

In [7]:
Xp = np.array([2., 1.])
d2p = C.sqdist(P, Xp)
p, f, idx = C.knn(P, nhan, ma, Xp[None, :], 3, return_index=True)
print('d² =', d2p.tolist(), '· láng giềng', [f'P{i + 1}' for i in idx[0]], '→', ['ham', 'spam'][p[0]], '· z(110) =', (110 - 100) / 20)
assert list(d2p) == [5, 1, 8, 13, 10]

d² = [5.0, 1.0, 8.0, 13.0, 10.0] · láng giềng ['P2', 'P1', 'P3'] → spam · z(110) = 0.5
